# Chapter 2, Session 3: Apache Iceberg — The Table Format

**Goal:** Understand what a "table format" actually does by using one.

We'll:
1. Create an Iceberg catalog connected to our MinIO
2. Create a table and write data
3. See ACID in action — atomic writes, no corrupt partial data
4. Time travel — query data as it was at a previous point
5. Schema evolution — add/rename/drop columns without rewriting data
6. Row-level updates and deletes (GDPR compliance)
7. Inspect the metadata to see what Iceberg tracks under the hood

## Step 1: Set Up Iceberg Catalog

An Iceberg **catalog** is the entry point — it knows where tables live and tracks their metadata.

Types of catalogs:
- **SQL catalog** (SQLite/PostgreSQL) — stores metadata in a database (we'll use this — simple, local)
- **Hive Metastore** — traditional Hadoop catalog, used by Spark
- **REST catalog** — HTTP service, used in production (Tabular, AWS Glue)
- **Glue catalog** — AWS managed, what Lake Formation uses

We'll use a **SQLite catalog** — it stores metadata in a local SQLite file. Simple for learning, same concepts as production.

In [ ]:
from pyiceberg.catalog.sql import SqlCatalog
import os

# Create a directory for our Iceberg warehouse
warehouse_path = os.path.abspath('../data/iceberg_warehouse')
os.makedirs(warehouse_path, exist_ok=True)

# Create catalog — this is the "brain" that tracks all tables
catalog = SqlCatalog(
    "local",
    **{
        "uri": f"sqlite:///{warehouse_path}/catalog.db",
        "warehouse": f"s3://iceberg-warehouse",
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin",
    }
)

print(f"Catalog created: {catalog.name}")
print(f"Warehouse: s3://iceberg-warehouse (MinIO)")
print(f"Metadata DB: {warehouse_path}/catalog.db")

In [ ]:
import boto3
from botocore.client import Config

# Create the warehouse bucket in MinIO
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3.create_bucket(Bucket='iceberg-warehouse')
    print("Created bucket: iceberg-warehouse")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print("Bucket already exists: iceberg-warehouse")

# Create a namespace (like a database/schema)
try:
    catalog.create_namespace("llm_data")
    print("Created namespace: llm_data")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Namespace already exists: llm_data")
    else:
        raise

print(f"\nNamespaces: {catalog.list_namespaces()}")

## Step 2: Create a Table and Write Data

Let's create an Iceberg table for our FineWeb documents and load data into it.

In [ ]:
import pyarrow as pa
import pandas as pd
import pickle

# Load our FineWeb data
with open('../data/fineweb_5000_samples.pkl', 'rb') as f:
    samples = pickle.load(f)

df = pd.DataFrame(samples)
# Add word_count for our quality filters
df['word_count'] = df['text'].str.split().str.len()

# Use first 1000 docs for faster operations
df_subset = df.head(1000).copy()
print(f"Working with {len(df_subset)} documents")
print(f"Columns: {list(df_subset.columns)}")

In [ ]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    StringType, LongType, DoubleType, NestedField
)

# Define the table schema
schema = Schema(
    NestedField(1, "id", StringType(), required=False),
    NestedField(2, "text", StringType(), required=False),
    NestedField(3, "url", StringType(), required=False),
    NestedField(4, "dump", StringType(), required=False),
    NestedField(5, "language", StringType(), required=False),
    NestedField(6, "language_score", DoubleType(), required=False),
    NestedField(7, "token_count", LongType(), required=False),
    NestedField(8, "word_count", LongType(), required=False),
)

# Create the table
try:
    catalog.drop_table("llm_data.documents")
except:
    pass

table = catalog.create_table(
    "llm_data.documents",
    schema=schema,
)

print(f"Created table: llm_data.documents")
print(f"\nSchema:")
for field in table.schema().fields:
    print(f"  {field.field_id}: {field.name} ({field.field_type})")

In [ ]:
# Write data to the Iceberg table
# Select only the columns that match our schema
write_df = df_subset[['id', 'text', 'url', 'dump', 'language', 
                       'language_score', 'token_count', 'word_count']].copy()

# Convert to PyArrow table
arrow_table = pa.Table.from_pandas(write_df)

# Write to Iceberg — this is an ATOMIC operation
table.append(arrow_table)

print(f"Wrote {len(write_df)} rows to llm_data.documents")
print(f"\nTable now has {len(table.scan().to_pandas())} rows")

**What just happened atomically:**
1. Iceberg wrote the data as Parquet files to MinIO (`s3://iceberg-warehouse/...`)
2. Created a **snapshot** — a record saying "this table consists of these Parquet files"
3. Updated the metadata to point to the new snapshot

If step 1 crashed halfway, no snapshot would be created → readers never see partial data. **This is ACID.**

## Step 3: Snapshots — The Core of Iceberg

Every write creates a new snapshot. Let's see them.

In [ ]:
from datetime import datetime

def show_snapshots(table):
    """Display all snapshots of an Iceberg table."""
    # Reload table to get latest metadata
    table = catalog.load_table("llm_data.documents")
    snapshots = table.metadata.snapshots
    current = table.metadata.current_snapshot_id
    
    print(f"Total snapshots: {len(snapshots)}")
    print(f"Current snapshot: {current}")
    print(f"\n{'#':<4} {'Snapshot ID':<22} {'Timestamp':<28} {'Operation':<12} {'Current'}")
    print('-' * 85)
    for i, snap in enumerate(snapshots):
        ts = datetime.fromtimestamp(snap.timestamp_ms / 1000).strftime('%Y-%m-%d %H:%M:%S')
        op = snap.summary.operation.value if snap.summary else 'unknown'
        is_current = '  ◄── YOU ARE HERE' if snap.snapshot_id == current else ''
        print(f"{i+1:<4} {snap.snapshot_id:<22} {ts:<28} {op:<12} {is_current}")
    return table

# Right now we should have 1 snapshot (the initial write)
table = show_snapshots(table)

In [ ]:
# Let's add more data to create a second snapshot
# Take the next 500 documents
extra_df = df.iloc[1000:1500][['id', 'text', 'url', 'dump', 'language',
                                'language_score', 'token_count', 'word_count']].copy()
table.append(pa.Table.from_pandas(extra_df))
print(f"Appended {len(extra_df)} more rows")

# Now we should have 2 snapshots
table = show_snapshots(table)

print(f"\nTotal rows now: {len(table.scan().to_pandas())}")

## Step 4: Time Travel

This is one of Iceberg's killer features. You can read the table as it was at any previous snapshot.

**Real-world use case:** "The model we trained 3 months ago performed better. What was the training data at that time?"

In [ ]:
# Read CURRENT version
current_df = table.scan().to_pandas()
print(f"Current version: {len(current_df)} rows")

# Read the FIRST snapshot (before we added the extra 500)
first_snapshot_id = table.metadata.snapshots[0].snapshot_id
old_df = table.scan(snapshot_id=first_snapshot_id).to_pandas()
print(f"Snapshot 1 (time travel): {len(old_df)} rows")

print(f"\nDifference: {len(current_df) - len(old_df)} rows were added between snapshots")
print(f"\nThis is TIME TRAVEL — you just read the table as it was before the second write.")
print(f"Both versions exist simultaneously. No data was copied or duplicated.")

**How does this work without duplicating data?**

```
Snapshot 1: points to [file1.parquet]              → 1000 rows
Snapshot 2: points to [file1.parquet, file2.parquet] → 1500 rows
```

`file1.parquet` is shared between both snapshots. Iceberg just tracks which files belong to which snapshot. Zero data duplication.

## Step 5: Schema Evolution

In production, schemas change. You realize you need a new column. Or a column name was wrong.

Without Iceberg: rewrite all your Parquet files (could be 50TB).
With Iceberg: update metadata only. Instant. Old files still work — new column just returns `null` for old rows.

In [ ]:
# Current schema
print("=== BEFORE Schema Evolution ===")
for field in table.schema().fields:
    print(f"  {field.field_id}: {field.name} ({field.field_type})")

In [ ]:
from pyiceberg.types import FloatType

# ADD a new column — quality_score
with table.update_schema() as update:
    update.add_column("quality_score", FloatType(), doc="Heuristic quality score 0-100")

print("Added column: quality_score")

# RENAME a column — 'dump' is unclear, rename to 'crawl_batch'
with table.update_schema() as update:
    update.rename_column("dump", "crawl_batch")

print("Renamed column: dump → crawl_batch")

# Reload and show new schema
table = catalog.load_table("llm_data.documents")
print("\n=== AFTER Schema Evolution ===")
for field in table.schema().fields:
    print(f"  {field.field_id}: {field.name} ({field.field_type})")

In [ ]:
# Read the data — notice quality_score is null for all existing rows
# (because the column didn't exist when those rows were written)
evolved_df = table.scan().to_pandas()
print(f"Rows: {len(evolved_df)}")
print(f"Columns: {list(evolved_df.columns)}")
print(f"\nquality_score values (should be all null):")
print(f"  Non-null count: {evolved_df['quality_score'].notna().sum()}")
print(f"  Null count: {evolved_df['quality_score'].isna().sum()}")
print(f"\n'crawl_batch' column (renamed from 'dump'):")
print(f"  Sample values: {evolved_df['crawl_batch'].head(3).tolist()}")

print("\nNO DATA FILES WERE REWRITTEN. Only metadata changed.")
print("Imagine doing this on 50TB — still instant with Iceberg.")

## Step 6: Row-Level Deletes (GDPR Compliance)

GDPR says: "A user requests deletion of their data. You must comply."

Without Iceberg: You'd have to read all Parquet files, filter out the rows, rewrite everything.
With Iceberg: Delete specific rows. Iceberg handles it efficiently.

In [ ]:
# Let's say we need to delete all documents from a specific URL domain
# (simulating a GDPR "right to be forgotten" request)

before_count = len(table.scan().to_pandas())
print(f"Before delete: {before_count} rows")

# Find a URL pattern to delete
sample_urls = evolved_df['url'].head(10).tolist()
print(f"\nSample URLs:")
for url in sample_urls[:5]:
    print(f"  {url}")

# Delete rows matching a specific URL
target_url = sample_urls[0]
print(f"\nDeleting rows with URL: {target_url}")

table.delete(delete_filter=f"url == '{target_url}'")

after_count = len(table.scan().to_pandas())
print(f"After delete: {after_count} rows")
print(f"Deleted: {before_count - after_count} rows")

In [ ]:
# Time travel still works — we can see the deleted rows in the old snapshot!
table = show_snapshots(table)

# The first snapshot still has all data
first_snap = table.metadata.snapshots[0].snapshot_id
old_data = table.scan(snapshot_id=first_snap).to_pandas()
print(f"\nSnapshot 1 still has: {len(old_data)} rows (including 'deleted' ones)")
print(f"Current version has: {len(table.scan().to_pandas())} rows")
print(f"\nThe data isn't physically deleted yet — just not referenced by the current snapshot.")
print(f"To physically remove it (for true GDPR compliance), you'd run 'expire_snapshots' + 'remove_orphan_files'.")

## Step 7: Inspect the Metadata — What Iceberg Tracks

Let's peek behind the curtain and see what Iceberg stores in MinIO.

In [ ]:
# List all files Iceberg created in MinIO
response = s3.list_objects_v2(Bucket='iceberg-warehouse', MaxKeys=200)

print("=== Files in iceberg-warehouse bucket ===")
print("Remember the three-level tree: metadata.json → manifest list → manifest files → data files\n")
metadata_json = []
manifest_lists = []
manifest_files = []
data_files = []

for obj in response.get('Contents', []):
    size_kb = obj['Size'] / 1024
    key = obj['Key']
    basename = key.split('/')[-1]
    
    if basename.endswith('.metadata.json'):
        metadata_json.append(key)
        print(f"  [LEVEL 1 - metadata.json] {key:<75} {size_kb:>6.1f} KB")
    elif basename.startswith('snap-'):
        manifest_lists.append(key)
        print(f"  [LEVEL 2 - manifest list] {key:<75} {size_kb:>6.1f} KB")
    elif basename.endswith('.avro'):
        manifest_files.append(key)
        print(f"  [LEVEL 3 - manifest file] {key:<75} {size_kb:>6.1f} KB")
    elif basename.endswith('.parquet'):
        data_files.append(key)
        print(f"  [DATA    - parquet file ] {key:<75} {size_kb:>6.1f} KB")
    else:
        print(f"  [OTHER                  ] {key:<75} {size_kb:>6.1f} KB")

print(f"\n=== Summary ===")
print(f"  Level 1 (metadata.json files): {len(metadata_json)}  ← one per schema/snapshot change")
print(f"  Level 2 (manifest lists):      {len(manifest_lists)}  ← one per snapshot")
print(f"  Level 3 (manifest files):      {len(manifest_files)}  ← lists of data files + column stats")
print(f"  Data (parquet files):          {len(data_files)}  ← your actual data")

In [ ]:
# Inspect the table's detailed metadata
table = catalog.load_table("llm_data.documents")
metadata = table.metadata

print("=== Table Metadata (Level 1: metadata.json) ===")
print(f"Table UUID: {metadata.table_uuid}")
print(f"Format version: {metadata.format_version}")
print(f"Location: {metadata.location}")
print(f"Last updated: {datetime.fromtimestamp(metadata.last_updated_ms / 1000)}")
print(f"Current snapshot: {metadata.current_snapshot_id}")
print(f"Total snapshots: {len(metadata.snapshots)}")

print(f"\n=== Schema History (field IDs are stable — renames don't break reads) ===")
for i, schema in enumerate(metadata.schemas):
    current = " ◄── CURRENT" if schema.schema_id == metadata.current_schema_id else ""
    print(f"  Schema {schema.schema_id}{current}:")
    for field in schema.fields:
        print(f"    field_id={field.field_id}: {field.name} ({field.field_type})")

print(f"\n=== Snapshot → Manifest List chain ===")
for snap in metadata.snapshots:
    ts = datetime.fromtimestamp(snap.timestamp_ms / 1000).strftime('%Y-%m-%d %H:%M:%S')
    current = " ◄── CURRENT" if snap.snapshot_id == metadata.current_snapshot_id else ""
    print(f"  Snapshot {snap.snapshot_id} [{ts}]{current}")
    print(f"    manifest-list: {snap.manifest_list}")
    if snap.summary:
        for k, v in snap.summary.additional_properties.items():
            print(f"    {k}: {v}")

### What you're seeing above is the three-level tree:

```
Level 1: metadata.json
   ├── schema history (field IDs stay stable across renames)
   ├── snapshot list (each with a manifest-list pointer)
   └── current-snapshot-id (the atomic pointer)
         │
         ▼
Level 2: snap-xxxx.avro (manifest list)
   └── lists manifest files + summary stats (added/deleted file counts)
         │
         ▼
Level 3: manifest-xxxx.avro (manifest files)
   └── lists actual Parquet files + per-file stats (row count, size, column min/max)
         │
         ▼
Data:    xxxx.parquet (your actual data — never modified by Iceberg)
```

**The entire ACID guarantee comes from one thing:** the catalog pointer to metadata.json is swapped atomically. Everything else is just immutable files on S3.

In [ ]:
# Dive into Level 3: Manifest files — what data files exist and their stats
print("=== Manifest Files (Level 3) — The Data File Registry ===\n")

current_snapshot = [s for s in metadata.snapshots if s.snapshot_id == metadata.current_snapshot_id][0]

# Get manifest entries for the current snapshot
manifests = table.inspect.manifests()
manifests_df = manifests.to_pandas()
print(f"Current snapshot has {len(manifests_df)} manifest(s):\n")
for _, m in manifests_df.iterrows():
    print(f"  Manifest: {m.get('path', 'N/A')}")
    print(f"    Added data files: {m.get('added_data_files_count', 'N/A')}")
    print(f"    Existing data files: {m.get('existing_data_files_count', 'N/A')}")
    print(f"    Deleted data files: {m.get('deleted_data_files_count', 'N/A')}")
    print()

# Get the actual data file listing
print("=== Data Files Referenced by Current Snapshot ===\n")
files = table.inspect.data_files()
files_df = files.to_pandas()
for _, f in files_df.iterrows():
    print(f"  File: {f.get('file_path', 'N/A')}")
    print(f"    Record count: {f.get('record_count', 'N/A')}")
    print(f"    File size: {f.get('file_size_in_bytes', 0) / 1024:.1f} KB")
    print(f"    File format: {f.get('file_format', 'N/A')}")
    print()

## Step 8: Column Pruning + Filtering with Iceberg

Iceberg pushes down column selection AND row filters to avoid reading unnecessary data.

In [ ]:
import time

# Full scan — read everything
start = time.time()
full = table.scan().to_pandas()
full_time = time.time() - start

# Column pruning — only read 2 columns
start = time.time()
pruned = table.scan(
    selected_fields=("token_count", "language_score")
).to_pandas()
pruned_time = time.time() - start

# Row filter — only read documents with 100+ words
start = time.time()
filtered = table.scan(
    row_filter="word_count >= 100",
    selected_fields=("text", "word_count", "url")
).to_pandas()
filtered_time = time.time() - start

print(f"{'Scan Type':<40} {'Rows':>8} {'Columns':>10} {'Time':>10}")
print('-' * 70)
print(f"{'Full scan (all columns, all rows)':<40} {len(full):>8} {len(full.columns):>10} {full_time:>9.3f}s")
print(f"{'Column pruning (2 columns only)':<40} {len(pruned):>8} {len(pruned.columns):>10} {pruned_time:>9.3f}s")
print(f"{'Row filter + column pruning':<40} {len(filtered):>8} {len(filtered.columns):>10} {filtered_time:>9.3f}s")

print(f"\nIceberg pushes these filters down to the Parquet reader level.")
print(f"At TB scale, this means reading 5GB instead of 500GB.")

## Summary

| Feature | What You Did | Why It Matters |
|---------|-------------|----------------|
| **Catalog** | Created SQLite catalog → MinIO | Entry point for all table operations |
| **ACID writes** | Appended data atomically | No corrupt partial files if pipeline crashes |
| **Snapshots** | Saw snapshot history | Every write is versioned automatically |
| **Time travel** | Read old snapshot | Reproduce exact training data from any point |
| **Schema evolution** | Added column, renamed column | Change schema on 50TB without rewriting data |
| **Row-level delete** | Deleted by URL filter | GDPR compliance without full rewrite |
| **Metadata inspection** | Browsed MinIO files | Iceberg = metadata layer on top of Parquet files |
| **Predicate pushdown** | Filtered rows + pruned columns | Read less data = faster + cheaper |

### How This Connects to Production

What we did locally with SQLite + MinIO, enterprises do with:

| Local | Production (enterprise) |
|------------|------------------------|
| SQLite catalog | AWS Glue Catalog / Hive Metastore |
| MinIO | AWS S3 / GCS / Azure Blob |
| PyIceberg | Spark + Iceberg / Trino + Iceberg |
| Manual snapshots | Automated via Spark jobs / Airflow DAGs |

**The concepts are identical. Only the scale changes.**